<a href="https://colab.research.google.com/github/nayanjha16/CodeGen-Implementations-May_26/blob/Group-6/Project_6_Codegen_Checkpoint_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CHECKPOINT‑1: COMPLETE CODING (GOOGLE COLAB READY)**

**#1 : Install Dependencies**

In [ ]:
!pip install transformers datasets accelerate sentencepiece torch -q


**#2 : Imports**

In [ ]:
import json
import os
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments


**#3 : Create Minimal Subset Datasets (Java, C#)**

In [ ]:
os.makedirs("data", exist_ok=True)

# NL -> Java
nl_java = [
    {
        "nl": "Write a function to compute factorial of a number.",
        "java": "public static int factorial(int n) {\n    if (n <= 1) return 1;\n    return n * factorial(n - 1);\n}\n"
    },
    {
        "nl": "Write a function to find the maximum element in an integer array.",
        "java": "public static int max(int[] arr) {\n    int m = arr[0];\n    for (int x : arr) if (x > m) m = x;\n    return m;\n}\n"
    }
]

with open("data/nl_java.jsonl", "w") as f:
    for s in nl_java:
        f.write(json.dumps(s) + "\n")

# Java -> C#
java_csharp = [
    {
        "java": "public static int max(int[] arr) {\n    int m = arr[0];\n    for (int x : arr) if (x > m) m = x;\n    return m;\n}\n",
        "csharp": "public static int Max(int[] arr) {\n    int m = arr[0];\n    foreach (int x in arr) if (x > m) m = x;\n    return m;\n}\n"
    },
    {
        "java": "public static int factorial(int n) {\n    if (n <= 1) return 1;\n    return n * factorial(n - 1);\n}\n",
        "csharp": "public static int Factorial(int n) {\n    if (n <= 1) return 1;\n    return n * Factorial(n - 1);\n}\n"
    }
]

with open("data/java_csharp.jsonl", "w") as f:
    for s in java_csharp:
        f.write(json.dumps(s) + "\n")

# Code -> Documentation
code_doc = [
    {
        "code": "public static int max(int[] arr) {\n    int m = arr[0];\n    for (int x : arr) if (x > m) m = x;\n    return m;\n}\n",
        "doc": "Returns the maximum value in the given integer array."
    },
    {
        "code": "public static int factorial(int n) {\n    if (n <= 1) return 1;\n    return n * factorial(n - 1);\n}\n",
        "doc": "Computes the factorial of a non-negative integer using recursion."
    }
]

with open("data/code_doc.jsonl", "w") as f:
    for s in code_doc:
        f.write(json.dumps(s) + "\n")


**#4 : Dataset Class**

In [ ]:
class TextPairDataset(Dataset):
    def __init__(self, path, tokenizer, source_key, target_key, max_length=512):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        with open(path, "r") as f:
            for line in f:
                obj = json.loads(line)
                src = obj[source_key]
                tgt = obj[target_key]
                text = f"<src>\n{src}\n</src>\n<tgt>\n{tgt}"
                self.samples.append(text)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text = self.samples[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        enc["labels"] = enc["input_ids"].clone()
        return {k: v.squeeze(0) for k, v in enc.items()}


In [ ]:
%%writefile data/java_csharp_50.jsonl
{"java":"public static int add(int a, int b) { return a + b; }","csharp":"public static int Add(int a, int b) { return a + b; }"}
{"java":"public static int subtract(int a, int b) { return a - b; }","csharp":"public static int Subtract(int a, int b) { return a - b; }"}
{"java":"public static int add(int a, int b) { return a + b; }","csharp":"public static int Add(int a, int b) { return a + b; }"}
{"java":"public static int subtract(int a, int b) { return a - b; }","csharp":"public static int Subtract(int a, int b) { return a - b; }"}
{"java":"public static int multiply(int a, int b) { return a * b; }","csharp":"public static int Multiply(int a, int b) { return a * b; }"}
{"java":"public static int divide(int a, int b) { return a / b; }","csharp":"public static int Divide(int a, int b) { return a / b; }"}
{"java":"public static int max(int[] arr) { int m = arr[0]; for(int x: arr) if(x > m) m = x; return m; }","csharp":"public static int Max(int[] arr) { int m = arr[0]; foreach(int x in arr) if(x > m) m = x; return m; }"}
{"java":"public static int min(int[] arr) { int m = arr[0]; for(int x: arr) if(x < m) m = x; return m; }","csharp":"public static int Min(int[] arr) { int m = arr[0]; foreach(int x in arr) if(x < m) m = x; return m; }"}
{"java":"public static int sumArray(int[] arr) { int s = 0; for(int x: arr) s += x; return s; }","csharp":"public static int SumArray(int[] arr) { int s = 0; foreach(int x in arr) s += x; return s; }"}
{"java":"public static int countEven(int[] arr) { int c = 0; for(int x: arr) if(x % 2 == 0) c++; return c; }","csharp":"public static int CountEven(int[] arr) { int c = 0; foreach(int x in arr) if(x % 2 == 0) c++; return c; }"}
{"java":"public static int factorial(int n) { if(n <= 1) return 1; return n * factorial(n-1); }","csharp":"public static int Factorial(int n) { if(n <= 1) return 1; return n * Factorial(n-1); }"}
{"java":"public static boolean isEven(int n) { return n % 2 == 0; }","csharp":"public static bool IsEven(int n) { return n % 2 == 0; }"}
{"java":"public static boolean isOdd(int n) { return n % 2 != 0; }","csharp":"public static bool IsOdd(int n) { return n % 2 != 0; }"}
{"java":"public static int abs(int n) { return n < 0 ? -n : n; }","csharp":"public static int Abs(int n) { return n < 0 ? -n : n; }"}
{"java":"public static int square(int n) { return n * n; }","csharp":"public static int Square(int n) { return n * n; }"}
{"java":"public static int cube(int n) { return n * n * n; }","csharp":"public static int Cube(int n) { return n * n * n; }"}
{"java":"public static int linearSearch(int[] arr, int key) { for(int i=0;i<arr.length;i++) if(arr[i]==key) return i; return -1; }","csharp":"public static int LinearSearch(int[] arr, int key) { for(int i=0;i<arr.Length;i++) if(arr[i]==key) return i; return -1; }"}
{"java":"public static int countOccurrences(int[] arr, int key) { int c=0; for(int x: arr) if(x==key) c++; return c; }","csharp":"public static int CountOccurrences(int[] arr, int key) { int c=0; foreach(int x in arr) if(x==key) c++; return c; }"}
{"java":"public static int reverseNumber(int n) { int r=0; while(n>0){ r=r*10+n%10; n/=10;} return r;}","csharp":"public static int ReverseNumber(int n) { int r=0; while(n>0){ r=r*10+n%10; n/=10;} return r;}"}
{"java":"public static boolean isPalindrome(int n) { return n == reverseNumber(n); }","csharp":"public static bool IsPalindrome(int n) { return n == ReverseNumber(n); }"}
{"java":"public static int power(int a, int b) { int r=1; for(int i=0;i<b;i++) r*=a; return r; }","csharp":"public static int Power(int a, int b) { int r=1; for(int i=0;i<b;i++) r*=a; return r; }"}
{"java":"public static int gcd(int a, int b) { while(b!=0){ int t=b; b=a%b; a=t;} return a;}","csharp":"public static int Gcd(int a, int b) { while(b!=0){ int t=b; b=a%b; a=t;} return a;}"}
{"java":"public static int lcm(int a, int b) { return a*b/gcd(a,b); }","csharp":"public static int Lcm(int a, int b) { return a*b/Gcd(a,b); }"}
{"java":"public static int countDigits(int n) { int c=0; while(n>0){c++; n/=10;} return c;}","csharp":"public static int CountDigits(int n) { int c=0; while(n>0){c++; n/=10;} return c;}"}
{"java":"public static int sumDigits(int n) { int s=0; while(n>0){s+=n%10; n/=10;} return s;}","csharp":"public static int SumDigits(int n) { int s=0; while(n>0){s+=n%10; n/=10;} return s;}"}
{"java":"public static boolean isPrime(int n) { if(n<=1) return false; for(int i=2;i*i<=n;i++) if(n%i==0) return false; return true;}","csharp":"public static bool IsPrime(int n) { if(n<=1) return false; for(int i=2;i*i<=n;i++) if(n%i==0) return false; return true;}"}
{"java":"public static int fibonacci(int n) { if(n<=1) return n; return fibonacci(n-1)+fibonacci(n-2);}","csharp":"public static int Fibonacci(int n) { if(n<=1) return n; return Fibonacci(n-1)+Fibonacci(n-2);}"}
{"java":"public static int sumFirstN(int n) { return n*(n+1)/2; }","csharp":"public static int SumFirstN(int n) { return n*(n+1)/2; }"}
{"java":"public static int productFirstN(int n) { int p=1; for(int i=1;i<=n;i++) p*=i; return p;}","csharp":"public static int ProductFirstN(int n) { int p=1; for(int i=1;i<=n;i++) p*=i; return p;}"}
{"java":"public static int lastDigit(int n) { return n%10; }","csharp":"public static int LastDigit(int n) { return n%10; }"}
{"java":"public static int firstDigit(int n) { while(n>=10) n/=10; return n;}","csharp":"public static int FirstDigit(int n) { while(n>=10) n/=10; return n;}"}
{"java":"public static int average(int[] arr) { int s=0; for(int x:arr) s+=x; return s/arr.length;}","csharp":"public static int Average(int[] arr) { int s=0; foreach(int x in arr) s+=x; return s/arr.Length;}"}
{"java":"public static int countPositive(int[] arr) { int c=0; for(int x:arr) if(x>0) c++; return c;}","csharp":"public static int CountPositive(int[] arr) { int c=0; foreach(int x in arr) if(x>0) c++; return c;}"}
{"java":"public static int countNegative(int[] arr) { int c=0; for(int x:arr) if(x<0) c++; return c;}","csharp":"public static int CountNegative(int[] arr) { int c=0; foreach(int x in arr) if(x<0) c++; return c;}"}
{"java":"public static int findIndex(int[] arr, int key) { for(int i=0;i<arr.length;i++) if(arr[i]==key) return i; return -1;}","csharp":"public static int FindIndex(int[] arr, int key) { for(int i=0;i<arr.Length;i++) if(arr[i]==key) return i; return -1;}"}
{"java":"public static int reverseArray(int[] arr) { int i=0,j=arr.length-1; while(i<j){ int t=arr[i]; arr[i]=arr[j]; arr[j]=t; i++; j--; } return 1;}","csharp":"public static int ReverseArray(int[] arr) { int i=0,j=arr.Length-1; while(i<j){ int t=arr[i]; arr[i]=arr[j]; arr[j]=t; i++; j--; } return 1;}"}
{"java":"public static int countZeroes(int[] arr) { int c=0; for(int x:arr) if(x==0) c++; return c;}","csharp":"public static int CountZeroes(int[] arr) { int c=0; foreach(int x in arr) if(x==0) c++; return c;}"}
{"java":"public static int findSmallestPositive(int[] arr) { int m=Integer.MAX_VALUE; for(int x:arr) if(x>0 && x<m) m=x; return m;}","csharp":"public static int FindSmallestPositive(int[] arr) { int m=int.MaxValue; foreach(int x in arr) if(x>0 && x<m) m=x; return m;}"}
{"java":"public static int findLargestNegative(int[] arr) { int m=Integer.MIN_VALUE; for(int x:arr) if(x<0 && x>m) m=x; return m;}","csharp":"public static int FindLargestNegative(int[] arr) { int m=int.MinValue; foreach(int x in arr) if(x<0 && x>m) m=x; return m;}"}
{"java":"public static int countPrimes(int[] arr) { int c=0; for(int x:arr) if(isPrime(x)) c++; return c;}","csharp":"public static int CountPrimes(int[] arr) { int c=0; foreach(int x in arr) if(IsPrime(x)) c++; return c;}"}
{"java":"public static int sumEven(int[] arr) { int s=0; for(int x:arr) if(x%2==0) s+=x; return s;}","csharp":"public static int SumEven(int[] arr) { int s=0; foreach(int x in arr) if(x%2==0) s+=x; return s;}"}
{"java":"public static int sumOdd(int[] arr) { int s=0; for(int x:arr) if(x%2!=0) s+=x; return s;}","csharp":"public static int SumOdd(int[] arr) { int s=0; foreach(int x in arr) if(x%2!=0) s+=x; return s;}"}
{"java":"public static int difference(int a, int b) { return a - b; }","csharp":"public static int Difference(int a, int b) { return a - b; }"}
{"java":"public static int product(int a, int b) { return a * b; }","csharp":"public static int Product(int a, int b) { return a * b; }"}
{"java":"public static int remainder(int a, int b) { return a % b; }","csharp":"public static int Remainder(int a, int b) { return a % b; }"}
{"java":"public static int doubleValue(int n) { return n * 2; }","csharp":"public static int DoubleValue(int n) { return n * 2; }"}
{"java":"public static int tripleValue(int n) { return n * 3; }","csharp":"public static int TripleValue(int n) { return n * 3; }"}


In [ ]:
!ls data


**#5 : Train Java → C# Translator (PL1 → PL2)**

In [46]:
MODEL = "Salesforce/codegen-350M-multi"

tokenizer_jc = AutoTokenizer.from_pretrained(MODEL)
model_jc = AutoModelForCausalLM.from_pretrained(MODEL)

# Fix: Set the padding token
tokenizer_jc.pad_token = tokenizer_jc.eos_token

# train_jc = TextPairDataset("data/java_csharp.jsonl", tokenizer_jc, "java", "csharp")
# val_jc   = TextPairDataset("data/java_csharp.jsonl", tokenizer_jc, "java", "csharp")

train_ds_jc = TextPairDataset("data/java_csharp_50.jsonl", tokenizer_jc, "java", "csharp")
val_ds_jc   = TextPairDataset("data/java_csharp_50.jsonl", tokenizer_jc, "java", "csharp")


args_jc = TrainingArguments(
    output_dir="outputs/java_to_csharp",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=5,
    eval_steps=10,
    save_steps=10,
    fp16=False, # Changed to False to prevent FP16 gradient unscaling error
    report_to="tensorboard",
    max_grad_norm=1.0,
)

trainer_jc = Trainer(
    model=model_jc,
    args=args_jc,
    train_dataset=train_jc,
    eval_dataset=val_jc,
)

trainer_jc.train()
trainer_jc.save_model("outputs/java_to_csharp/final")
tokenizer_jc.save_pretrained("outputs/java_to_csharp/final")

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('outputs/java_to_csharp/final/tokenizer_config.json',
 'outputs/java_to_csharp/final/tokenizer.json')

In [47]:
%%writefile data/code_doc_50.jsonl
{"code":"public static int add(int a, int b) { return a + b; }","doc":"Returns the sum of two integers."}
{"code":"public static int subtract(int a, int b) { return a - b; }","doc":"Computes the difference between two integers."}
{"code":"public static int multiply(int a, int b) { return a * b; }","doc":"Multiplies two integers and returns the result."}
{"code":"public static int divide(int a, int b) { return a / b; }","doc":"Performs integer division of a by b."}
{"code":"public static int max(int[] arr) { int m=arr[0]; for(int x:arr) if(x>m) m=x; return m;}","doc":"Finds and returns the maximum value in the array."}
{"code":"public static int min(int[] arr) { int m=arr[0]; for(int x:arr) if(x<m) m=x; return m;}","doc":"Returns the smallest element in the array."}
{"code":"public static int sumArray(int[] arr) { int s=0; for(int x:arr) s+=x; return s;}","doc":"Calculates the sum of all elements in the array."}
{"code":"public static int countEven(int[] arr) { int c=0; for(int x:arr) if(x%2==0) c++; return c;}","doc":"Counts how many even numbers are present in the array."}
{"code":"public static int factorial(int n) { if(n<=1) return 1; return n*factorial(n-1);}","doc":"Computes the factorial of n using recursion."}
{"code":"public static boolean isEven(int n) { return n%2==0; }","doc":"Checks whether the given number is even."}
{"code":"public static boolean isOdd(int n) { return n%2!=0; }","doc":"Determines if the input number is odd."}
{"code":"public static int abs(int n) { return n<0?-n:n; }","doc":"Returns the absolute value of the input integer."}
{"code":"public static int square(int n) { return n*n; }","doc":"Computes the square of the given number."}
{"code":"public static int cube(int n) { return n*n*n; }","doc":"Returns the cube of the input integer."}
{"code":"public static int linearSearch(int[] arr, int key) { for(int i=0;i<arr.length;i++) if(arr[i]==key) return i; return -1;}","doc":"Searches for a key in the array and returns its index or -1 if not found."}
{"code":"public static int countOccurrences(int[] arr, int key) { int c=0; for(int x:arr) if(x==key) c++; return c;}","doc":"Counts how many times the key appears in the array."}
{"code":"public static int reverseNumber(int n) { int r=0; while(n>0){ r=r*10+n%10; n/=10;} return r;}","doc":"Reverses the digits of the given integer."}
{"code":"public static boolean isPalindrome(int n) { return n==reverseNumber(n); }","doc":"Checks whether the number reads the same forward and backward."}
{"code":"public static int power(int a, int b) { int r=1; for(int i=0;i<b;i++) r*=a; return r;}","doc":"Computes a raised to the power b using iterative multiplication."}
{"code":"public static int gcd(int a, int b) { while(b!=0){ int t=b; b=a%b; a=t;} return a;}","doc":"Finds the greatest common divisor of two integers using the Euclidean algorithm."}
{"code":"public static int lcm(int a, int b) { return a*b/gcd(a,b); }","doc":"Computes the least common multiple of two integers."}
{"code":"public static int countDigits(int n) { int c=0; while(n>0){c++; n/=10;} return c;}","doc":"Counts the number of digits in the given integer."}
{"code":"public static int sumDigits(int n) { int s=0; while(n>0){s+=n%10; n/=10;} return s;}","doc":"Returns the sum of all digits of the input number."}
{"code":"public static boolean isPrime(int n) { if(n<=1) return false; for(int i=2;i*i<=n;i++) if(n%i==0) return false; return true;}","doc":"Checks whether the number is a prime number."}
{"code":"public static int fibonacci(int n) { if(n<=1) return n; return fibonacci(n-1)+fibonacci(n-2);}","doc":"Computes the nth Fibonacci number."}
{"code":"public static int sumFirstN(int n) { return n*(n+1)/2; }","doc":"Calculates the sum of the first n natural numbers."}
{"code":"public static int productFirstN(int n) { int p=1; for(int i=1;i<=n;i++) p*=i; return p;}","doc":"Computes the product of the first n natural numbers."}
{"code":"public static int lastDigit(int n) { return n%10; }","doc":"Returns the last digit of the number."}
{"code":"public static int firstDigit(int n) { while(n>=10) n/=10; return n;}","doc":"Returns the first digit of the number."}
{"code":"public static int average(int[] arr) { int s=0; for(int x:arr) s+=x; return s/arr.length;}","doc":"Calculates the average of elements in the array."}
{"code":"public static int countPositive(int[] arr) { int c=0; for(int x:arr) if(x>0) c++; return c;}","doc":"Counts positive numbers in the array."}
{"code":"public static int countNegative(int[] arr) { int c=0; for(int x:arr) if(x<0) c++; return c;}","doc":"Counts negative numbers in the array."}
{"code":"public static int findIndex(int[] arr, int key) { for(int i=0;i<arr.length;i++) if(arr[i]==key) return i; return -1;}","doc":"Finds the index of a key in the array, returns -1 if not found."}
{"code":"public static int reverseArray(int[] arr) { int i=0,j=arr.length-1; while(i<j){ int t=arr[i]; arr[i]=arr[j]; arr[j]=t; i++; j--; } return 1;}","doc":"Reverses the elements of the array in place."}
{"code":"public static int countZeroes(int[] arr) { int c=0; for(int x:arr) if(x==0) c++; return c;}","doc":"Counts occurrences of zero in the array."}
{"code":"public static int findSmallestPositive(int[] arr) { int m=Integer.MAX_VALUE; for(int x:arr) if(x>0 && x<m) m=x; return m;}","doc":"Finds the smallest positive number in the array."}
{"code":"public static int findLargestNegative(int[] arr) { int m=Integer.MIN_VALUE; for(int x:arr) if(x<0 && x>m) m=x; return m;}","doc":"Finds the largest negative number in the array."}
{"code":"public static int countPrimes(int[] arr) { int c=0; for(int x:arr) if(isPrime(x)) c++; return c;}","doc":"Counts prime numbers in the array."}
{"code":"public static int sumEven(int[] arr) { int s=0; for(int x:arr) if(x%2==0) s+=x; return s;}","doc":"Sums all even numbers in the array."}
{"code":"public static int sumOdd(int[] arr) { int s=0; for(int x:arr) if(x%2!=0) s+=x; return s;}","doc":"Sums all odd numbers in the array."}
{"code":"public static int difference(int a, int b) { return a - b; }","doc":"Calculates the difference between two integers."}
{"code":"public static int product(int a, int b) { return a * b; }","doc":"Computes the product of two integers."}
{"code":"public static int remainder(int a, int b) { return a % b; }","doc":"Calculates the remainder of a divided by b."}
{"code":"public static int doubleValue(int n) { return n * 2; }","doc":"Returns double the value of the input integer."}
{"code":"public static int tripleValue(int n) { return n * 3; }","doc":"Returns triple the value of the input integer."}

Overwriting data/code_doc_50.jsonl


In [48]:
!ls data

code_doc_50.jsonl  java_csharp_50.jsonl  nl_java.jsonl
code_doc.jsonl	   java_csharp.jsonl


**# 6 : Train Documentation Generator**

In [49]:
tokenizer_doc = AutoTokenizer.from_pretrained(MODEL)
model_doc = AutoModelForCausalLM.from_pretrained(MODEL)

# Fix: Set the padding token
tokenizer_doc.pad_token = tokenizer_doc.eos_token

# train_doc = TextPairDataset("data/code_doc.jsonl", tokenizer_doc, "code", "doc")
# val_doc   = TextPairDataset("data/code_doc.jsonl", tokenizer_doc, "code", "doc")

train_ds_doc = TextPairDataset("data/code_doc_50.jsonl", tokenizer_doc, "code", "doc")
val_ds_doc   = TextPairDataset("data/code_doc_50.jsonl", tokenizer_doc, "code", "doc")


args_doc = TrainingArguments(
    output_dir="outputs/code_doc",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=5,
    eval_steps=10,
    save_steps=10,
    fp16=False, # Changed to False for training stability
    report_to="none",
)

trainer_doc = Trainer(
    model=model_doc,
    args=args_doc,
    train_dataset=train_ds_doc, # Corrected variable name
    eval_dataset=val_ds_doc,     # Corrected variable name
)

trainer_doc.train()
trainer_doc.save_model("outputs/code_doc/final")
tokenizer_doc.save_pretrained("outputs/code_doc/final")

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
5,1188.137988
10,4404.754297
15,0.000000
20,0.000000
25,0.000000
30,0.000000
35,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('outputs/code_doc/final/tokenizer_config.json',
 'outputs/code_doc/final/tokenizer.json')

**# 7 : Inference: NL → Java**

In [ ]:
tokenizer_nl_java = AutoTokenizer.from_pretrained(MODEL)
model_nl_java = AutoModelForCausalLM.from_pretrained(MODEL)

"""
 def nl_to_java(nl):
    prompt = f"Write a Java function for the following requirement:\n{nl}\n\nJava code:\n"
    inputs = tokenizer_nl_java(prompt, return_tensors="pt")
    output = model_nl_java.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        top_p=0.9,
        temperature=0.7
    )
    return tokenizer_nl_java.decode(output[0], skip_special_tokens=True)[len(prompt):]

"""
def nl_to_java(nl):
    prompt = f"""
Convert the following description into a simple Java function.
Rules:
- Use only primitive types
- No classes except 'Solution'
- No helper methods
- No recursion
- No external libraries
- Keep code under 10 lines

Description:
{nl}

Java code:
public class Solution {{
    public static int solve(int n) {{
        // Write code here
"""
    inputs = tokenizer_nl_java(prompt, return_tensors="pt")
    output = model_nl_java.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        top_p=0.9,
        temperature=0.7
    )
    full = tokenizer_nl_java.decode(output[0], skip_special_tokens=True)
    return full.split("Java code:")[-1].strip()



**# 8 : Inference: Java → C#**

In [ ]:
tokenizer_jc_inf = AutoTokenizer.from_pretrained("outputs/java_to_csharp/final")
model_jc_inf = AutoModelForCausalLM.from_pretrained("outputs/java_to_csharp/final")

def java_to_csharp(java_code):
    prompt = f"<src>\n{java_code}\n</src>\n<tgt>\n"
    inputs = tokenizer_jc_inf(prompt, return_tensors="pt")
    output = model_jc_inf.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False, # Changed to greedy decoding for stability
        top_p=0.9,
        temperature=0.7
    )
    return tokenizer_jc_inf.decode(output[0], skip_special_tokens=True).split("<tgt>")[-1]

**# 9 : Inference: Documentation Generation**

In [ ]:
from transformers import GenerationConfig

tokenizer_doc_inf = AutoTokenizer.from_pretrained("outputs/code_doc/final")
model_doc_inf = AutoModelForCausalLM.from_pretrained("outputs/code_doc/final")

def generate_doc(code):
    prompt = f"<src>\n{code}\n</src>\n<tgt>\n"
    inputs = tokenizer_doc_inf(prompt, return_tensors="pt")

    # Ensure greedy decoding by directly passing do_sample=False and num_beams=1
    # Removed top_p and temperature as they are only relevant for sampling.
    output = model_doc_inf.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False, # Explicitly set for greedy decoding
        num_beams=1,     # Explicitly set num_beams to 1 for greedy decoding
        pad_token_id=tokenizer_doc_inf.pad_token_id,
        eos_token_id=tokenizer_doc_inf.eos_token_id
    )
    return tokenizer_doc_inf.decode(output[0], skip_special_tokens=True).split("<tgt>")[-1]

**# 10 : End‑to‑End Pipeline (Checkpoint‑1 Demo)**

In [53]:
"""
nl = "Write a function to compute factorial of a number."

java_code = nl_to_java(nl)
csharp_code = java_to_csharp(java_code)
doc_java = generate_doc(java_code)
doc_csharp = generate_doc(csharp_code)

print("=== Natural Language ===\n", nl)
print("\n=== Java Code ===\n", java_code)
print("\n=== C# Code ===\n", csharp_code)
print("\n=== Documentation (Java) ===\n", doc_java)
print("\n=== Documentation (C#) ===\n", doc_csharp)
"""

nl = "Write a function to compute factorial of a number."

# 1. NL → Java
java_code = nl_to_java(nl)
print("=== Java Code ===")
print(java_code)

# 2. Java → C#
csharp_code = java_to_csharp(java_code)
print("\n=== C# Code ===")
print(csharp_code)

# 3. Documentation (Java)
doc_java = generate_doc(java_code)
print("\n=== Documentation (Java) ===")
print(doc_java)

# 4. Documentation (C#)
doc_csharp = generate_doc(csharp_code)
print("\n=== Documentation (C#) ===")
print(doc_csharp)


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== Java Code ===
public class Solution {
    public static int solve(int n) {
        // Write code here
        if (n == 1) {
            return 1;
        } else if (n == 0) {
            return 0;
        } else {
            int sum = 1;
            for (int i = 2; i <= n; i++) {
                sum = sum * i;
            }
            return sum;
        }
    }
}

=== C# Code ===



=== Documentation (Java) ===

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

=== Documentation (C#) ===

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
